# M4 — Forecast-based inventory & replenishment export

Uses **weekly ML forecasts** + latest inventory snapshot to compute:

- Target stock = lead-time demand + safety stock
- Recommended order quantity
- Estimated PO cost

Outputs feed the Streamlit dashboard and FastAPI service.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.forecast import run_modelling
from src.pipeline import run_data_foundation
from src.risk import run_risk_and_recommendations

clean, _ = run_data_foundation(persist=False)
modelling = run_modelling(clean["sales_daily"], persist=True)
outputs = run_risk_and_recommendations(
    clean["inventory_snapshots"],
    clean["sales_daily"],
    clean["sku_master"],
    modelling["forecasts"],
    persist=True,
)

print("Backtest WAPE — RF:", round(modelling["metrics"]["random_forest_wape_pct"], 2), "%")
print("Backtest WAPE — Seasonal naive:", round(modelling["metrics"]["seasonal_naive_wape_pct"], 2), "%")
print("Replenishment SKUs:", len(outputs["recommendations"]))

Backtest WAPE — RF: 10.05 %
Backtest WAPE — Seasonal naive: 11.51 %
Replenishment SKUs: 5


## Replenishment table (dashboard input)

In [2]:
rec = outputs["recommendations"]
rec

,SKU,Product_Name,Current_Stock,On_Order,Forecast,Lead_Time_Days,Recommended_Units,Estimated_Cost,Recommendation
11,SKU012,Product 012,47,60,155.960,12,177,222469.53,Order 177 units
16,SKU017,Product 017,95,26,94.005,12,149,998785.74,Order 149 units
30,SKU031,Product 031,93,40,101.290,13,84,529254.60,Order 84 units
9,SKU010,Product 010,83,70,102.170,13,59,229454.54,Order 59 units
39,SKU040,Product 040,67,29,84.690,9,49,253284.43,Order 49 units


In [3]:
print("Total recommended units:", rec["Recommended_Units"].sum())
print("Total estimated cost (INR):", f"{rec['Estimated_Cost'].sum():,.2f}")
print("Saved to:", ROOT / "data/forecast_inventory_recommendations.csv")

Total recommended units: 518
Total estimated cost (INR): 2,233,248.84
Saved to: /home/gulshan-yadav/Documents/foresight_project/data/forecast_inventory_recommendations.csv


## Risk score preview

In [4]:
risk = outputs["risk"]
print(risk["Risk_Status"].value_counts())
risk[risk["Risk_Status"].str.contains("Stockout|Overstock")][
    ["SKU", "Product_Name", "Days_of_Cover", "Lead_Time_Days", "Risk_Status"]
].head(10)

Risk_Status
Overstock Risk          39
High Risk (Stockout)     5
Healthy                  5
Reorder Attention        1
Name: count, dtype: int64


,SKU,Product_Name,Days_of_Cover,Lead_Time_Days,Risk_Status
11,SKU012,Product 012,4.081077,12,High Risk (Stockout)
39,SKU040,Product 040,6.112981,9,High Risk (Stockout)
30,SKU031,Product 031,7.030660,13,High Risk (Stockout)
16,SKU017,Product 017,8.506578,12,High Risk (Stockout)
9,SKU010,Product 010,8.942886,13,High Risk (Stockout)
25,SKU026,Product 026,10.123239,5,Overstock Risk
8,SKU009,Product 009,15.298507,7,Overstock Risk
12,SKU013,Product 013,17.376387,4,Overstock Risk
44,SKU045,Product 045,17.504873,4,Overstock Risk
47,SKU048,Product 048,18.371467,3,Overstock Risk
